# 5. Conclusion

## 5.1 Key results

Every metric below is computed from `data/m3_*` when the book is built; chapters [3](statistics.ipynb) and
[4](results.ipynb) give the full breakdowns.

In [ ]:
import pandas as pd

import report_data as rd

rd.check()
ds = rd.data_summary()
lbl = rd.labels()
met = rd.metrics()
run = rd.run_log()
req = rd.requirements()
hfe = rd.hf_export()

s = ds["summary"]
m = lbl["manual"]
a = lbl["auto"]
avm = lbl["auto_vs_manual"]
ho = lbl["auto_vs_manual_held_out"]
f_val = met["final"]["valid"]["all"]
f_test = met["final"]["test"]["all"]
f_tr = met["final"]["train"]["all"]
n_pass = sum(c["status"] == "PASS" for c in req["checks"])
exp_run = hfe["runs"][-1]
tot_rows = sum(exp_run["tables"][t]["source_rows"] for t in exp_run["tables"])

rows = [
    ("Locked upstream parser", f"`{ds['parser_version']}` ({ds['papers']:,} papers)"),
    ("Eligible publications", f"{s['eligible']:,} of {s['papers']:,} ({100 * s['eligible_share']:.1f}%)"),
    ("Manual ground truth", f"{m['papers']} papers (5/member), {m['entries']:,} entries, {m['matched']:,} matched pairs"),
    ("Automatic labels", f"{a['papers']:,} papers ({100 * lbl['auto_share_outside_manual']:.1f}% outside manual), "
                         f"{a['entries']:,} entries, {a['matched']:,} pairs"),
    ("Auto vs manual (all 20 / 8 held-out)",
     f"P={avm['precision']:.4f}, R={avm['recall']:.4f}, F1={avm['f1']:.4f} / "
     f"P={ho['precision']:.4f}, R={ho['recall']:.4f}, F1={ho['f1']:.4f}"),
    ("Candidate pairs & features",
     f"{run['runs'][0]['steps']['features']['pairs']:,} pairs × 25 features ({rd.hm(run['runs'][0]['steps']['features']['seconds'])})"),
    ("Chosen model", f"`{met['chosen']['config']}` (`tau = {met['chosen']['tau']}`, classical CPU-only)"),
    ("Official MRR@5 (`skip`): valid / test / train OOF",
     f"{f_val['mrr5_skip']:.4f} / {f_test['mrr5_skip']:.4f} / {f_tr['mrr5_skip']:.5f}"),
    ("Hit@1: valid / test", f"{f_val['hit1']:.4f} / {f_test['hit1']:.4f}"),
    ("`mrr5_zero` (valid / test)", f"{f_val['mrr5_zero']:.4f} / {f_test['mrr5_zero']:.4f}"),
    ("Requirement audit (`24_analysis`)", f"{n_pass} / {len(req['checks'])} PASS"),
    ("Hugging Face export", f"6 tables × 4 members, {tot_rows:,} total rows, 0 rows outside ranges"),
    ("End-to-end M3 run wall time (28 Nov)", f"{rd.hm(run['runs'][0]['seconds'])} ({run['runs'][0]['seconds']:,.1f} s)"),
]
rd.table(pd.DataFrame(rows, columns=["Result", "Value"]), "Table 5.1. Milestone 3 at a glance.")

## 5.2 What worked

- **Locking the parser to 13.9 and fixing REVTeX early.** Re-parsing the 190 REVTeX papers before finalizing
  `data/m3/` eliminated 10,727 macro-corrupted titles/authors while verifying that all 20 manual-label papers
  remained byte-identical.
- **Leak-free labelling and evaluation.** Drawing the manual test and validation papers *before* tuning the
  automatic matcher, excluding both active and replaced (`manual_old/`) manual papers from automatic
  labelling, and using publication-grouped 5-fold CV with out-of-fold train predictions prevented data leakage
  at every stage.
- **Unified cleaning and matrix-context features.** Cleaning BibTeX entries and Semantic Scholar references
  with the exact same functions made simple string and TF-IDF features (`f_raw_tfidf`, `f_title_tfidf_char`,
  `f_col_rank`) extraordinarily sharp (AUC > 0.995), while Milestone 2 citing sentences (`f_ctx_tfidf`) added
  an independent topical signal (AUC 0.760).
- **Automated auditing and schema enforcement.** Running `24_analysis` (23/23 PASS) and `check_export()`
  before uploading guaranteed that every Parquet table conforms to the statement's Arrow schema, has zero
  cross-range leakage, and streams cleanly on Hugging Face.

## 5.3 Limitations

- **MRR@5 saturation under clean in-paper matching.** Because citations carry explicit identifiers (37.0%
  arXiv ID, 22.8% DOI), near-verbatim titles (mean similarity 0.991), and a median candidate pool of only 18
  arXiv references per paper, `baseline`, `logreg`, and `LGBMRanker` all tie at 1.0000 on `valid` and `test`.
  The standard metric cannot separate stronger models from the baseline on this clean split.
- **Small validation and test publication counts.** Per the statement's group-scaled split ($k$ manual + $k$
  automatic), `valid` and `test` hold 8 papers each (318 and 267 matched entries). We mitigated this with
  5-fold `GroupKFold` CV on all 3,757 training papers and 16-paper LOPO on the manual set, though the test
  split itself remains small.
- **Upstream ineligibility (24.8% of papers).** 4,960 of 20,000 papers cannot enter Milestone 3 because
  Milestone 1/2 produced no BibTeX entries (mostly PDF-only submissions and the 2,231 `missing_bbl` versions)
  or because Semantic Scholar returned zero arXiv references.
- **NIL detection is harder than ranking.** When an application does not know in advance whether an entry
  cites an arXiv paper, thresholding the top candidate score trades precision against recall (precision 0.991
  at recall 0.901 near the true 27.6% NIL rate on `valid`).

## 5.4 What we would do next

1. **Harder stress-test benchmark (proposed, not yet run).** Randomly mask identifiers (`arxiv_hint`, `doi`,
   `url`) and perturb or truncate titles in a Colab run of `23_model` on `pairs.parquet` to measure where
   `logreg` and `LGBMRanker` (using author, year, matrix-context and M2 citation-context features) pull ahead
   of the unlearned `baseline`.
2. **Dedicated NIL classifier.** Train a binary calibrated classifier using the row-constant confidence
   features (`f_margin`, `f_has_arxiv_hint`, `f_venue_arxiv`, `f_n_candidates`) so the pipeline can abstain
   cleanly on non-arXiv citations.
3. **Recover `missing_bbl` bibliographies.** Re-downloading the 2,231 `missing_bbl` versions with their `.bbl`
   files (~2 hours) and re-running the ready `.bbl` converter in `12_bibtex` would recover thousands of
   missing bibliography entries and raise Milestone 3 eligibility substantially.